In [1]:
# Version X
# should calculate with float64 for precision
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

# The t-ohlcv data was pre-fetched
folder_path = find_project_root() / "data" / "bigData"

symbol = "BTCUSDT"
tf = "4h"

PATH =folder_path / f"{symbol}-regimed-{tf}.json"

with open(PATH) as f:
    arr = json.load(f)

# Binance API fetched columns
cols = ["timestamp", "open", "high", "low", "close", "vol", "close-timestamp", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol","checksum"]

df = pd.DataFrame(arr, columns=cols)
del arr  
gc.collect() # free the Python var, preserve RAM

# Halve memory: float64 -> float32 for price/volume columns
df[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]] = df[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]].astype("float32")

print(f"5m Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 160 MB

# sort by timestamp
df = df.sort_values('timestamp').reset_index(drop=True)
df.head()

5m Shape: (11989, 12) | Memory: 1.22 MB


,timestamp,open,high,low,close,vol,close-timestamp,quote-vol,taker#,taker-buy-basevol,taker-buy-quotevol,checksum
0,1600560000000,11080.639648,11080.639648,10899.330078,10948.809570,6383.454590,1600574399999,70069336.0,103942.0,2967.512695,32577704.0,0
1,1600574400000,10948.799805,10995.309570,10933.000000,10962.610352,4171.369629,1600588799999,45714088.0,70269.0,1980.400269,21703114.0,0
2,1600588800000,10962.599609,10984.000000,10925.269531,10951.169922,5002.634277,1600603199999,54818136.0,75565.0,2409.657227,26405682.0,0
3,1600603200000,10951.169922,10958.179688,10775.000000,10834.580078,9520.058594,1600617599999,103340552.0,151849.0,4628.103027,50246096.0,0
4,1600617600000,10834.620117,10903.769531,10723.000000,10871.030273,7652.687012,1600631999999,82997944.0,138938.0,3702.844971,40164504.0,0


In [2]:
# Drop no use cols
drop_cols = ["close-timestamp", "taker-buy-quotevol","checksum", "quote-vol"]
df.drop(columns=drop_cols, inplace=True)

In [3]:
print(df.columns)
df.head()

Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'taker#',
       'taker-buy-basevol'],
      dtype='object')


,timestamp,open,high,low,close,vol,taker#,taker-buy-basevol
0,1600560000000,11080.639648,11080.639648,10899.330078,10948.809570,6383.454590,103942.0,2967.512695
1,1600574400000,10948.799805,10995.309570,10933.000000,10962.610352,4171.369629,70269.0,1980.400269
2,1600588800000,10962.599609,10984.000000,10925.269531,10951.169922,5002.634277,75565.0,2409.657227
3,1600603200000,10951.169922,10958.179688,10775.000000,10834.580078,9520.058594,151849.0,4628.103027
4,1600617600000,10834.620117,10903.769531,10723.000000,10871.030273,7652.687012,138938.0,3702.844971


# Save Data

In [4]:
# Export marker to json visulize the data on Chart
# save export to jsonl
# Export marker to json visulize the data on Chart
# save export to jsonl

save_path = find_project_root() / "data" / "mlData" / "raw"
save_path.mkdir(parents=True, exist_ok=True)

out_path = save_path / f"{symbol}-{tf}-vX.jsonl"
df.to_json(out_path, orient="records", lines=True)
print(f"Saved {len(df):,} rows → {out_path}")

Saved 11,989 rows → /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-candle-science/data/mlData/raw/BTCUSDT-4h-vX.jsonl


In [ ]:
print(df.columns)